# Transformations in `moro`

This notebook presents examples of how to use functions from the `moro.transformations` submodule related to:

1. Rotation matrices in $SO(3)$.
2. Proper Euler angles.
3. Homogeneous transformation matrices in $SE(3)$.

The `moro` library is based on **SymPy**, so these functions can work with both numeric values and symbolic expressions.

## Setup

We import SymPy for symbolic manipulation and the `moro.transformations` submodule as `mt`.

In [ ]:
import sympy as sp
from sympy import Matrix, pi
from IPython.display import Math, display

import moro.transformations as mt

sp.init_printing()

## Summary of functions used

| Topic | Main functions | Description |
|---|---|---|
| Elementary rotations | `rotx`, `roty`, `rotz`, `rot` | Build $3\times3$ rotation matrices about the $x$, $y$, and $z$ axes. |
| Proper Euler angles | `eul2rot`, `rot2eul` | Convert between a proper Euler angle triplet and a rotation matrix. |
| Homogeneous transformations | `htmtra`, `htmrot`, `rot2htm`, `rt2htm`, `htm2rot`, `htm2tra`, `invhtm`, `dh` | Build, decompose, and invert $4\times4$ homogeneous matrices. |

# 1. Rotation matrices

A rotation matrix $R \in SO(3)$ represents the orientation of a coordinate frame or the rotation of a vector in space. It must satisfy:

$$
R^T R = I, \qquad \det(R)=1.
$$

In `moro.transformations`, the elementary rotations are available as:

$$
R_x(\theta), \quad R_y(\theta), \quad R_z(\theta).
$$

By default, angles are interpreted in **radians**. If `deg=True` is used, they are interpreted in **degrees**.

In [ ]:
theta = sp.symbols("theta")

Rx = mt.rotx(theta)
Ry = mt.roty(theta)
Rz = mt.rotz(theta)

display(Math(r"R_x(\theta) ="))
display(Rx)
display(Math(r"R_y(\theta) ="))
display(Ry)
display(Math(r"R_z(\theta) ="))
display(Rz)

You can also use the general function `rot(theta, axis="z", deg=False)`, specifying the axis with a string (`"x"`, `"y"`, or `"z"`).

In [ ]:
Rz_90 = mt.rot(90, axis="z", deg=True)
Rx_pi_2 = mt.rot(pi/2, axis="x")

print("Rotation of 90 degrees about z:")
display(Rz_90)

print("Rotation of pi/2 radians about x:")
display(Rx_pi_2)

## Verifying that a matrix belongs to $SO(3)$

For an ideal rotation matrix, the inverse is equal to the transpose:

$$
R^{-1}=R^T.
$$

Let us verify this for a symbolic rotation about $z$.

In [ ]:
R = mt.rotz(theta)

orthogonality = sp.simplify(R.T * R)
determinant = sp.simplify(R.det())

print("R.T * R =")
display(orthogonality)

print("det(R) =")
display(determinant)

## Composition of rotations

Rotations are composed through matrix multiplication. For example, if we first apply a rotation about $z$ and then one about $x$, the composed matrix is:

$$
R = R_x(\alpha) R_z(\beta).
$$

The multiplication order is important because, in general, 3D rotations do not commute.

In [ ]:
alpha, beta = sp.symbols("alpha beta")

R_xz = mt.rotx(alpha) * mt.rotz(beta)
R_zx = mt.rotz(beta) * mt.rotx(alpha)

print("R_x(alpha) * R_z(beta):")
display(R_xz)

print("R_z(beta) * R_x(alpha):")
display(R_zx)

print("Are they equal?")
display(sp.simplify(R_xz - R_zx) == sp.zeros(3))

## Rotating a vector

If $p$ is a column vector, the rotated vector is obtained as:

$$
p' = R p.
$$

In [ ]:
p = Matrix([1, 0, 0])
R = mt.rotz(90, deg=True)
p_rotated = R * p

print("Original vector:")
display(p)

print("Vector rotated 90 degrees about z:")
display(p_rotated)

# 2. Proper Euler angles

`moro.transformations` implements **proper Euler angles**, i.e., sequences where the first and third axes are the same:

| Sequence | Matrix product |
|---|---|
| `"xyx"` | $R_x(\phi) R_y(\theta) R_x(\psi)$ |
| `"xzx"` | $R_x(\phi) R_z(\theta) R_x(\psi)$ |
| `"yxy"` | $R_y(\phi) R_x(\theta) R_y(\psi)$ |
| `"yzy"` | $R_y(\phi) R_z(\theta) R_y(\psi)$ |
| `"zxz"` | $R_z(\phi) R_x(\theta) R_z(\psi)$ |
| `"zyz"` | $R_z(\phi) R_y(\theta) R_z(\psi)$ |

The convention used by `eul2rot` is:

$$
R = R_a(\phi) R_b(\theta) R_a(\psi),
$$

with column vectors and active rotations. Tait-Bryan sequences such as `"xyz"` are not supported by these functions.

## From Euler angles to a rotation matrix

Use `eul2rot(phi, theta, psi, seq="zxz", deg=False)`.

In [ ]:
R_euler = mt.eul2rot(30, 45, 60, seq="zxz", deg=True)

print("Rotation matrix for ZXZ Euler angles = (30 deg, 45 deg, 60 deg):")
display(sp.N(R_euler, 4))

The previous result should match the explicit multiplication of the three elementary rotations in the `"zxz"` sequence.

In [ ]:
R_manual = mt.rotz(30, deg=True) * mt.rotx(45, deg=True) * mt.rotz(60, deg=True)

print("Does eul2rot match the manual composition?")
display(sp.simplify(R_euler - R_manual) == sp.zeros(3))

## From a rotation matrix to Euler angles

The function `rot2eul(R, seq="zxz", deg=False, tol=1e-9)` returns a list of solutions. In the general case, the same matrix may have two equivalent Euler angle triplets:

$$
[(\phi_1,\theta_1,\psi_1), (\phi_2,\theta_2,\psi_2)].
$$

At singularities, the function returns one representative solution with $\psi=0$.

In [ ]:
solutions = mt.rot2eul(R_euler, seq="zxz", deg=True)

print("ZXZ Euler solutions in degrees:")
for sol in solutions:
    display(tuple(sp.N(angle, 6) for angle in sol))

We can reconstruct the original matrix with any of the returned solutions.

In [ ]:
tol = 1e-9
for i, (phi, theta, psi) in enumerate(solutions, start=1):
    R_reconstructed = mt.eul2rot(phi, theta, psi, seq="zxz", deg=True)
    error = R_reconstructed - R_euler
    max_error = max(abs(float(sp.N(value))) for value in error)
    print(f"Solution {i}: maximum error = {max_error:.2e}")
    display(max_error < tol)

## Symbolic use with Euler angles

Because `moro` is based on SymPy, we can also build Euler rotation matrices with symbolic variables.

In [ ]:
phi, theta, psi = sp.symbols("phi theta psi")

R_zyz_symbolic = mt.eul2rot(phi, theta, psi, seq="zyz")

display(Math(r"R_{zyz}(\phi,\theta,\psi) ="))
display(R_zyz_symbolic)

# 3. Homogeneous transformation matrices

A homogeneous matrix represents orientation and position simultaneously:

$$
T = \begin{bmatrix}
R & p \\
0\;0\;0 & 1
\end{bmatrix},
$$

where $R \in SO(3)$ is the rotation matrix and $p \in \mathbb{R}^3$ is the translation vector.

These matrices allow us to transform homogeneous points:

$$
\begin{bmatrix}p'\\1\end{bmatrix} =
T \begin{bmatrix}p\\1\end{bmatrix}.
$$

## Pure translation with `htmtra`

`htmtra(x=0, y=0, z=0)` builds a homogeneous transformation with identity rotation and translation $(x,y,z)$.

In [ ]:
T_translation = mt.htmtra(x=2, y=-1, z=3)

display(T_translation)

## Pure rotation with `htmrot`

`htmrot(theta, axis="z", deg=False)` builds a homogeneous matrix whose rotational part is an elementary rotation and whose translation is zero.

In [ ]:
T_rotation = mt.htmrot(90, axis="z", deg=True)

display(T_rotation)

## Building a transformation from $R$ and $p$

With `rt2htm(R, p)`, we can build directly:

$$
T = \begin{bmatrix} R & p \\ 0 & 1 \end{bmatrix}.
$$

You can also convert a rotation matrix $R$ into a homogeneous matrix with zero translation using `rot2htm(R)`.

In [ ]:
R = mt.roty(45, deg=True)
p = Matrix([2, 0, 1])

T = mt.rt2htm(R, p)
T_from_rotation_only = mt.rot2htm(R)

print("T built with rt2htm(R, p):")
display(sp.N(T, 4))

print("T built with rot2htm(R):")
display(sp.N(T_from_rotation_only, 4))

## Extracting rotation and translation

Given a homogeneous matrix `T`, we can recover its blocks with:

- `htm2rot(T)` to obtain $R$.
- `htm2tra(T)` to obtain $p$.

In [ ]:
R_extracted = mt.htm2rot(T)
p_extracted = mt.htm2tra(T)

print("Extracted rotation:")
display(sp.N(R_extracted, 4))

print("Extracted translation:")
display(p_extracted)

## Structured inverse of a homogeneous transformation

If

$$
T = \begin{bmatrix} R & p \\ 0 & 1 \end{bmatrix},
$$

then its inverse is:

$$
T^{-1} = \begin{bmatrix}
R^T & -R^T p \\
0 & 1
\end{bmatrix}.
$$

The function `invhtm(T)` uses this structure instead of a generic matrix inverse.

In [ ]:
T_inv = mt.invhtm(T)

print("T_inv:")
display(sp.N(T_inv, 4))

print("T_inv * T:")
display(sp.N(sp.simplify(T_inv * T), 4))

## Composition and point transformation

Matrix multiplication composes homogeneous transformations. In the next example, we build a transformation as a translation followed by a rotation, and apply it to a homogeneous point.

In [ ]:
T_composed = mt.htmtra(x=1, y=2, z=0) * mt.htmrot(90, axis="z", deg=True)

p_homogeneous = Matrix([1, 0, 0, 1])
p_transformed = T_composed * p_homogeneous

print("Composed T:")
display(T_composed)

print("Transformed point:")
display(p_transformed)

## Denavit-Hartenberg matrix with `dh`

The function `dh(a, alpha, d, theta)` generates a homogeneous matrix using the standard Denavit-Hartenberg parameters:

| Parameter | Meaning |
|---|---|
| $a$ | link length |
| $\alpha$ | link twist |
| $d$ | link offset |
| $\theta$ | joint angle |

The resulting matrix has the form:

$$
A_i =
\begin{bmatrix}
\cos\theta & -\sin\theta\cos\alpha & \sin\theta\sin\alpha & a\cos\theta \\
\sin\theta & \cos\theta\cos\alpha & -\cos\theta\sin\alpha & a\sin\theta \\
0 & \sin\alpha & \cos\alpha & d \\
0 & 0 & 0 & 1
\end{bmatrix}.
$$

In [ ]:
a, alpha, d, theta = sp.symbols("a alpha d theta")

A = mt.dh(a, alpha, d, theta)
display(A)

In [ ]:
A_num = mt.dh(a=1, alpha=pi/2, d=0.5, theta=pi/4)

display(A_num)

# Conclusions

In this notebook, we saw how to:

- Build rotation matrices with `rotx`, `roty`, `rotz`, and `rot`.
- Compose rotations and apply them to vectors.
- Convert between proper Euler angles and rotation matrices with `eul2rot` and `rot2eul`.
- Build, compose, decompose, and invert homogeneous matrices with functions from `moro.transformations`.
- Generate a homogeneous matrix from Denavit-Hartenberg parameters with `dh`.

Future versions of this notebook may add examples for axis-angle representation (`axa2rot`, `rot2axa`) and skew-symmetric matrices (`skew`).